# 라이브러리 및 데이터

In [143]:
from pathlib import Path
import pandas as pd
import numpy as np

import plotly.graph_objects as go

In [144]:
DATA_DIR = Path("../open")
TRAIN_DIR = DATA_DIR/"train"

ldaps_train = pd.read_csv(TRAIN_DIR/"ldaps_train.csv", encoding = "utf-8-sig", parse_dates = ['forecast_kst_dtm'])
gfs_train = pd.read_csv(TRAIN_DIR/"gfs_train.csv", encoding = "utf-8-sig", parse_dates = ['forecast_kst_dtm'])
scada_unison = pd.read_csv(TRAIN_DIR/"scada_unison_train.csv", encoding = "utf-8-sig", parse_dates = ['kst_dtm'])
scada_vestas = pd.read_csv(TRAIN_DIR/"scada_vestas_train.csv", encoding = "utf-8-sig", parse_dates = ['kst_dtm'])

In [145]:
scada_unison

,kst_dtm,unison_wtg01_power_kw10m,unison_wtg02_power_kw10m,unison_wtg03_power_kw10m,unison_wtg04_power_kw10m,unison_wtg05_power_kw10m,unison_wtg01_ws,unison_wtg02_ws,unison_wtg03_ws,unison_wtg04_ws,unison_wtg05_ws,unison_wtg01_wd,unison_wtg02_wd,unison_wtg03_wd,unison_wtg04_wd,unison_wtg05_wd
0,2023-01-01 00:10:00,706.0,702.0,0.0,705.0,707.0,13.73,15.03,16.42,13.97,14.57,-79.005327,-69.946609,-124.739623,-90.938062,-130.941880
1,2023-01-01 00:20:00,706.0,702.0,0.0,684.0,708.0,13.61,14.76,16.51,14.96,14.34,-77.479172,-68.192593,-122.671765,-87.976756,-131.394959
2,2023-01-01 00:30:00,708.0,702.0,0.0,408.0,706.0,14.59,14.70,16.41,12.68,13.55,-75.624282,-67.050591,-122.046216,-87.009944,-132.164142
3,2023-01-01 00:40:00,699.0,384.0,0.0,702.0,706.0,14.62,15.64,15.28,12.46,14.37,-76.608166,-66.836232,-122.675858,-88.533341,-132.392295
4,2023-01-01 00:50:00,610.0,471.0,0.0,698.0,706.0,14.20,15.36,14.79,14.16,14.95,-76.231935,-65.485185,-123.190201,-88.352211,-132.589051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105259,2024-12-31 23:20:00,0.0,700.0,700.0,600.0,700.0,NaN,11.35,11.53,11.86,12.59,-45.299988,-66.061753,-119.040238,-83.344987,-131.276571
105260,2024-12-31 23:30:00,0.0,700.0,700.0,600.0,700.0,NaN,11.12,12.00,10.79,12.57,-45.299988,-66.411235,-119.996131,-83.457039,-131.012316
105261,2024-12-31 23:40:00,0.0,700.0,700.0,600.0,700.0,NaN,11.60,12.98,11.82,12.53,-45.299988,-65.952024,-119.740193,-82.488376,-130.168194
105262,2024-12-31 23:50:00,0.0,700.0,700.0,600.0,700.0,NaN,11.31,11.93,10.98,11.65,-45.299988,-63.950665,-119.637973,-83.609951,-129.090439


# LDAPS, GFS의 grid_id별 격자 & SCADA 터빈 위치 비교
- 그룹별로 어떤 grid_id를 중요하게 봐야하는지 고려
- KPX Group1 : VESTAS 1 ~ 6호기
- KPX Group2 : VESTAS 7 ~ 12호기
- KPX Group3 : UNISON 1 ~ 5호기

## 1. 발전기 좌표를 위도와 경도로 변환
- latitude : 위도
- longitude : 경도

In [146]:
# 구글 기준 좌표를 위도와 경도로 변환하는 함수
import re

def dms_str_to_decimal(dms_str):
    deg, minu, sec, direction = re.match(
        r"(\d+)[°]\s*(\d+)['′]\s*([\d.]+)[\"″]\s*([NSEW])", dms_str
    ).groups()
    decimal = float(deg) + float(minu) / 60 + float(sec) / 3600
    if direction in ['S', 'W']:
        decimal *= -1
    return decimal

In [147]:
# info.xlsx에 있는 구글 기준 발전기 좌표 -> 위도와 경도로 변환
raw = pd.read_excel(DATA_DIR/"info.xlsx", sheet_name = "info", header=None)
header_row_idx = raw[raw.iloc[:, 1] == '단계'].index[0]
info = raw.iloc[header_row_idx + 1:].copy()
info.columns = raw.iloc[header_row_idx]
info = info.iloc[:, 1:]
info = info.reset_index(drop = True)

def parse_coord(coord_str):
    lat_str, lon_str = coord_str.strip().split(' ')
    lat = dms_str_to_decimal(lat_str)
    lon = dms_str_to_decimal(lon_str)
    return pd.Series({'latitude': lat, 'longitude': lon})

info[['latitude', 'longitude']] = info['좌표(Google)'].apply(parse_coord)

info.loc[[1, 2, 3, 4, 5], 'KPX그룹'] = 1
info.loc[[7, 8, 9, 10, 11], 'KPX그룹'] = 2
info.loc[[13, 14, 15, 16], 'KPX그룹'] = 3

In [148]:
info

3,단계,명칭,제작사,모델명,호기,좌표(Google),KPX그룹,Hub Height(m),Rotor Diameter(m),설비용량(MW),그룹설비용량(MW),latitude,longitude
0,1,태백가덕산,VESTAS,V126,1,"37°16'55.61""N 128°57'02.10""E",1,117,126,3.6,21.6,37.282114,128.950583
1,1,태백가덕산,VESTAS,V126,2,"37°17'04.05""N 128°56'58.35""E",1,117,126,3.6,NaN,37.284458,128.949542
2,1,태백가덕산,VESTAS,V126,3,"37°17'11.49""N 128°56'58.99""E",1,117,126,3.6,NaN,37.286525,128.949719
3,1,태백가덕산,VESTAS,V126,4,"37°17'23.11""N 128°57'03.68""E",1,117,126,3.6,NaN,37.289753,128.951022
4,1,태백가덕산,VESTAS,V126,5,"37°17'28.20""N 128°57'15.58""E",1,117,126,3.6,NaN,37.291167,128.954328
5,1,태백가덕산,VESTAS,V126,6,"37°17'19.48""N 128°57'24.96""E",1,117,126,3.6,NaN,37.288744,128.956933
6,1,태백가덕산,VESTAS,V126,7,"37°17'16.20""N 128°57'34.67""E",2,117,126,3.6,21.6,37.287833,128.959631
7,1,태백가덕산,VESTAS,V126,8,"37°17'11.29""N 128°57'47.24""E",2,117,126,3.6,NaN,37.286469,128.963122
8,1,태백가덕산,VESTAS,V126,9,"37°17'00.97""N 128°57'57.44""E",2,117,126,3.6,NaN,37.283603,128.965956
9,1,태백가덕산,VESTAS,V126,10,"37°16'52.77""N 128°58'04.18""E",2,117,126,3.6,NaN,37.281325,128.967828


In [149]:
# 위도, 경도를 포함한 데이터 새로 저장
info.to_csv("../data/info_include_lat_lon.csv", index = False, encoding = "utf-8-sig")

## 2. 그룹별 발전기 좌표와 LDAPS, GFS 좌표 시각화

In [150]:
# LDAPS 데이터에서 grid_id 별 위도, 경도 좌표 추출
ldaps_grid = ldaps_train[['grid_id', 'latitude', 'longitude']].drop_duplicates()

# GFS 데이터에서 grid_id 별 위도, 경도 좌표 추출
gfs_grid = gfs_train[['grid_id', 'latitude', 'longitude']].drop_duplicates()

In [151]:
fig = go.Figure()

# LDAPS 격자 (16개)
fig.add_trace(go.Scattermap(
    lat=ldaps_grid['latitude'],
    lon=ldaps_grid['longitude'],
    mode='markers+text',
    marker=dict(size=10, color='blue'),
    text=ldaps_grid['grid_id'].astype(str),
    textposition='top center',
    name='LDAPS grid'
))

# GFS 격자 (9개)
fig.add_trace(go.Scattermap(
    lat=gfs_grid['latitude'],
    lon=gfs_grid['longitude'],
    mode='markers+text',
    marker=dict(size=10, color='green'),
    text=gfs_grid['grid_id'].astype(str),
    textposition='top center',
    name='GFS grid'
))

# KPX 그룹 1, 2, 3 터빈 위치 (그룹별 색상 다르게)
group_colors = {1: 'red', 2: 'orange', 3: 'purple'}

for group, color in group_colors.items():
    kpx_turbines = info[info['KPX그룹'] == group]
    fig.add_trace(go.Scattermap(
        lat=kpx_turbines['latitude'],
        lon=kpx_turbines['longitude'],
        mode='markers',
        marker=dict(size=12, color=color, symbol='circle'),
        text=kpx_turbines['명칭'] + ' ' + kpx_turbines['호기'].astype(str),
        name=f'KPX Group {group} turbines'
    ))

# 모든 포인트의 위경도 범위를 계산해서 중심/줌 자동 설정
all_lat = pd.concat([ldaps_grid['latitude'], gfs_grid['latitude'], info['latitude']])
all_lon = pd.concat([ldaps_grid['longitude'], gfs_grid['longitude'], info['longitude']])

center_lat = all_lat.mean()
center_lon = all_lon.mean()

# 위경도 범위가 좁을수록 zoom을 높게
lat_range = all_lat.max() - all_lat.min()
lon_range = all_lon.max() - all_lon.min()
max_range = max(lat_range, lon_range)
zoom = 13 - max_range * 20  # 범위에 따라 대략적으로 조정, 필요시 계수 튜닝

fig.update_layout(
    map=dict(
        style='open-street-map',
        center=dict(lat=center_lat, lon=center_lon),
        zoom=zoom
    ),
    margin=dict(l=0, r=0, t=30, b=0),
    height=700,
    width=700,
    title='LDAPS / GFS 격자 및 KPX 그룹별 터빈 위치'
)
fig.show()

## 3. 이상치 등 처리

### SCADA Vestas - 극단값 오류
- vestas_wtgXX_power_kw10m 값 중 몇 개 정도가 이상치로 판단됨.
- 이 이상치가 발생한 날짜와 train_labels.csv의 group1/group2 결측 구간이 일치함.
- 즉 우연이 아니라 실제 설비/통신 장애 시점에 센서 오류인 듯.

### LDAPS 기상 데이터 - 물리적으로 불가능한 값
- 상대습도 (heightAboveGround_2_r) > 100%인 행이 16,649개
    - NWP 모델의 과포화 아티팩트로 흔한 현상이지만, feature로 사용할 때는 100으로 클리핑 권장.

- 이슬점(dpt)이 기온(t)보다 높은 행이 6,398개
    - 물리적으로 불가능 (이슬점은 기온을 넘을 수 없음.)
    - 습도 계산 시 같이 발생하는 것으로 보이며 클리핑 및 보정이 필요

- 운량(hcc, mcc, lcc) 값이 1을 살짝 초과하는 값이 수백 건
    - 부동소수점 반올림 오차
    - 1로 클리핑하면 됨.

- surface_0_lsm이 전체 행에서 항상 1
    - 분산이 0인 상수 컬럼이라 feature로는 의미 없음. (제거 대상)

In [152]:
# SCADA Vestas 극단값 전처리
RATED_KW_VESTAS = 3600   # V126 정격용량(kW)
vestas_power_cols = [c for c in scada_vestas.columns if 'power' in c]

for c in vestas_power_cols:
    invalid = (scada_vestas[c] < -50) | (scada_vestas[c] > RATED_KW_VESTAS * 1.2)
    scada_vestas.loc[invalid, c] = np.nan

print(scada_vestas[vestas_power_cols].isna().sum())

vestas_wtg01_power_kw10m     74
vestas_wtg02_power_kw10m     70
vestas_wtg03_power_kw10m     88
vestas_wtg04_power_kw10m     44
vestas_wtg05_power_kw10m     72
vestas_wtg06_power_kw10m     60
vestas_wtg07_power_kw10m     68
vestas_wtg08_power_kw10m    102
vestas_wtg09_power_kw10m     52
vestas_wtg10_power_kw10m     68
vestas_wtg11_power_kw10m     98
vestas_wtg12_power_kw10m     72
dtype: int64


In [153]:
# LDAPS 물리적 이상치 전처리
## 상대습도 100% 초과 -> 클리핑
ldaps_train['heightAboveGround_2_r'] = ldaps_train['heightAboveGround_2_r'].clip(upper = 100)

## 이슬점 > 기온 -> 기온으로 클리핑
mask = ldaps_train['heightAboveGround_2_dpt'] > ldaps_train['heightAboveGround_2_t']
ldaps_train.loc[mask, 'heightAboveGround_2_dpt'] = ldaps_train.loc[mask, 'heightAboveGround_2_t']

## 운량 1 초과 -> 클리핑
cloud_cols = ['etc_0_hcc', 'etc_0_mcc', 'etc_0_lcc', 'etc_0_VLCDC']
ldaps_train[cloud_cols] = ldaps_train[cloud_cols].clip(upper = 1)

## 상수 컬럼 제거
ldaps_train = ldaps_train.drop(columns = ['surface_0_lsm'])

## 4. KPX 그룹별 LDAPS, GFS grid_id 상관관계 분석

In [154]:
train_labels = pd.read_csv(TRAIN_DIR/'train_labels.csv', encoding = 'utf-8-sig', 
                           parse_dates = ['kst_dtm'])

In [155]:
# LDAPS : 50m 평균 u/v 성분을 만들고 그 크기로 풍속을 계산
# LDAPS : 117m에 가까운 50m를 선택
# GFS : 100m 선택
# 물리적으로 허브높이에 가까운 변수를 각 데이터셋에서 골라 사용함.
u_50m = (ldaps_train['heightAboveGround_50_50MUmax'] + ldaps_train['heightAboveGround_50_50MUmin']) / 2
v_50m = (ldaps_train['heightAboveGround_50_50MVmax'] + ldaps_train['heightAboveGround_50_50MVmin']) / 2
ldaps_train['ws_50m'] = np.sqrt(u_50m**2 + v_50m**2)
gfs_train['ws_100m'] = np.sqrt(gfs_train['heightAboveGround_100_100u']**2 + gfs_train['heightAboveGround_100_100v']**2)

In [156]:
# 시간을 행, grid_id를 열로 바꿈
# 그래야 grid_id 별로 발전량과의 상관관계를 각각 계산할 수 있음.
ldaps_wide = ldaps_train.pivot(index='forecast_kst_dtm', columns='grid_id', values='ws_50m')
ldaps_wide.columns = [f'ldaps_grid{c}' for c in ldaps_wide.columns]
gfs_wide = gfs_train.pivot(index='forecast_kst_dtm', columns='grid_id', values='ws_100m')
gfs_wide.columns = [f'gfs_grid{c}' for c in gfs_wide.columns]

merged = ldaps_wide.join(gfs_wide).merge(train_labels.set_index('kst_dtm'), left_index=True, right_index=True, how='inner')

In [158]:
capacity = {'kpx_group_1': 21600, 'kpx_group_2': 21600, 'kpx_group_3': 21000}
grid_cols = list(ldaps_wide.columns) + list(gfs_wide.columns)

top_grids_per_group = {}
for group, cap in capacity.items():
    valid = merged.dropna(subset=[group])
    valid = valid[valid[group] >= cap * 0.1]  # 평가 기준(설비용량 10% 이상)과 동일하게 필터링
    corr = valid[grid_cols].corrwith(valid[group], method='spearman').sort_values(ascending=False)
    top_grids_per_group[group] = corr.head(5)
    print(group, corr.head(5).round(3).to_dict())

kpx_group_1 {'ldaps_grid13': 0.702, 'ldaps_grid8': 0.691, 'ldaps_grid7': 0.686, 'ldaps_grid12': 0.684, 'ldaps_grid16': 0.678}
kpx_group_2 {'ldaps_grid13': 0.701, 'ldaps_grid8': 0.695, 'ldaps_grid7': 0.688, 'ldaps_grid12': 0.683, 'ldaps_grid16': 0.675}
kpx_group_3 {'ldaps_grid13': 0.662, 'ldaps_grid8': 0.658, 'ldaps_grid7': 0.652, 'ldaps_grid12': 0.647, 'ldaps_grid16': 0.637}


-> 세 그룹 모두 grid_id : 13, 8, 7, 12, 16 순으로 상관계수(풍속과 발전량) 순위 동일

## 5. 파워커브 학습 (전처리 데이터 기반)

In [160]:
from sklearn.isotonic import IsotonicRegression

def aggregate_scada_hourly(df, power_cols, ws_cols):
    df = df.copy()
    df['hour'] = df['kst_dtm'].dt.floor('h')
    records = []
    for i, (pc, wc) in enumerate(zip(power_cols, ws_cols)):
        t = df[['hour', pc, wc]].rename(columns={pc: 'power_kwh', wc: 'ws'})
        t['turbine_id'] = i
        records.append(t)
    long_df = pd.concat(records, ignore_index=True)
    hourly = long_df.groupby(['hour', 'turbine_id']).agg(
        power_kwh=('power_kwh', lambda x: x.sum(min_count=6)),  # 10분 발전량 합산 = 시간당 kWh
        ws=('ws', 'mean')
    ).reset_index()
    return hourly.dropna()

vestas_ws_cols = [c for c in scada_vestas.columns if c.endswith('_ws')]
vestas_hourly = aggregate_scada_hourly(scada_vestas, vestas_power_cols, vestas_ws_cols)  # 1번에서 이미 극단값 제거됨

unison_power_cols = [c for c in scada_unison.columns if 'power' in c]
unison_ws_cols = [c for c in scada_unison.columns if c.endswith('_ws')]
unison_hourly = aggregate_scada_hourly(scada_unison, unison_power_cols, unison_ws_cols)

def fit_power_curve(hourly_df, rated_kw):
    x, y = hourly_df['ws'].values, hourly_df['power_kwh'].clip(0, rated_kw).values
    curve = IsotonicRegression(y_min=0, y_max=rated_kw, out_of_bounds='clip')
    curve.fit(x, y)
    return curve

vestas_curve = fit_power_curve(vestas_hourly, rated_kw=3600)
unison_curve = fit_power_curve(unison_hourly, rated_kw=4200)

# LDAPS, GFS의 feature별 발전량과의 관계 살피기
- 어떤 feature 또는 행을 선택할지 고민

In [161]:
# ---------- 1. 터빈 배치로부터 능선(주축) 방향 계산 (PCA) ----------
lat0, lon0 = info['latitude'].mean(), info['longitude'].mean()
x = (info['longitude'] - lon0) * np.cos(np.radians(lat0)) * 111320
y = (info['latitude'] - lat0) * 110540
coords = np.column_stack([x, y])
coords = coords - coords.mean(axis=0)
eigvals, eigvecs = np.linalg.eigh(np.cov(coords.T))
ridge_dir = eigvecs[:, np.argmax(eigvals)]          # 터빈 배치 주축(능선 방향)
perp_dir = np.array([-ridge_dir[1], ridge_dir[0]])  # 능선을 가로지르는 방향

def project_wind(u, v, direction):
    return u * direction[0] + v * direction[1]

In [162]:
# ---------- 2. 기본 풍속 & 물리 기반 파생변수 ----------
ldaps_train['ws_10m'] = np.sqrt(ldaps_train['heightAboveGround_10_10u']**2 + ldaps_train['heightAboveGround_10_10v']**2)
u_50m = (ldaps_train['heightAboveGround_50_50MUmax'] + ldaps_train['heightAboveGround_50_50MUmin']) / 2
v_50m = (ldaps_train['heightAboveGround_50_50MVmax'] + ldaps_train['heightAboveGround_50_50MVmin']) / 2
ldaps_train['ws_50m'] = np.sqrt(u_50m**2 + v_50m**2)
ldaps_train['ws_50m_cubed'] = ldaps_train['ws_50m']**3
ldaps_train['wind_shear'] = ldaps_train['ws_50m'] - ldaps_train['ws_10m']
ldaps_train['dewpoint_dep'] = ldaps_train['heightAboveGround_2_t'] - ldaps_train['heightAboveGround_2_dpt']
ldaps_train['air_density'] = ldaps_train['surface_0_sp'] / (287.05 * ldaps_train['heightAboveGround_2_t'])
ldaps_train['ws_50m_density_adj'] = ldaps_train['ws_50m_cubed'] * ldaps_train['air_density']
ldaps_train['ws_50m_ridge_perp'] = project_wind(u_50m, v_50m, perp_dir)     # 능선 횡단 성분 (핵심)
ldaps_train['ws_50m_ridge_along'] = project_wind(u_50m, v_50m, ridge_dir)   # 능선 방향 성분 (참고)

gfs_train['ws_10m'] = np.sqrt(gfs_train['heightAboveGround_10_10u']**2 + gfs_train['heightAboveGround_10_10v']**2)
gfs_train['ws_80m'] = np.sqrt(gfs_train['heightAboveGround_80_u']**2 + gfs_train['heightAboveGround_80_v']**2)
gfs_train['ws_100m'] = np.sqrt(gfs_train['heightAboveGround_100_100u']**2 + gfs_train['heightAboveGround_100_100v']**2)
gfs_train['ws_pbl'] = np.sqrt(gfs_train['planetaryBoundaryLayer_0_u']**2 + gfs_train['planetaryBoundaryLayer_0_v']**2)
gfs_train['ws_100m_cubed'] = gfs_train['ws_100m']**3
gfs_train['wind_shear'] = gfs_train['ws_100m'] - gfs_train['ws_10m']
gfs_train['dewpoint_dep'] = gfs_train['heightAboveGround_2_2t'] - gfs_train['heightAboveGround_2_2d']
gfs_train['air_density'] = gfs_train['surface_0_sp'] / (287.05 * gfs_train['heightAboveGround_2_2t'])
gfs_train['ws_100m_density_adj'] = gfs_train['ws_100m_cubed'] * gfs_train['air_density']
gfs_train['ws_100m_ridge_perp'] = project_wind(gfs_train['heightAboveGround_100_100u'], gfs_train['heightAboveGround_100_100v'], perp_dir)

In [163]:
# ---------- 3. grid 대표값(평균) + 공간 표준편차 집계 ----------
top_ldaps_grids = [13, 8, 7, 12, 16]
ldaps_top = ldaps_train[ldaps_train['grid_id'].isin(top_ldaps_grids)]

ldaps_cols = ['ws_10m', 'ws_50m', 'ws_50m_cubed', 'wind_shear', 'dewpoint_dep',
              'air_density', 'ws_50m_density_adj', 'ws_50m_ridge_perp', 'ws_50m_ridge_along', 'etc_0_blh']
ldaps_mean = ldaps_top.groupby('forecast_kst_dtm')[ldaps_cols].mean().add_prefix('ldaps_')
ldaps_std = ldaps_top.groupby('forecast_kst_dtm')['ws_50m'].std().rename('ldaps_ws_50m_spatial_std')

gfs_cols = ['ws_10m', 'ws_80m', 'ws_100m', 'ws_pbl', 'ws_100m_cubed', 'wind_shear',
            'dewpoint_dep', 'air_density', 'ws_100m_density_adj', 'ws_100m_ridge_perp']
gfs_mean = gfs_train.groupby('forecast_kst_dtm')[gfs_cols].mean().add_prefix('gfs_')
gfs_std = gfs_train.groupby('forecast_kst_dtm')['ws_100m'].std().rename('gfs_ws_100m_spatial_std')

feat_table = ldaps_mean.join([ldaps_std, gfs_mean, gfs_std], how='inner').sort_index()

In [164]:
# ---------- 4. 시간/모델간 파생변수 ----------
feat_table['ldaps_ws_50m_ramp'] = feat_table['ldaps_ws_50m'].diff()
feat_table['gfs_ws_100m_ramp'] = feat_table['gfs_ws_100m'].diff()
feat_table['ws_10m_ldaps_gfs_diff'] = feat_table['ldaps_ws_10m'] - feat_table['gfs_ws_10m']
feat_table['ldaps_blh_shear_interact'] = feat_table['ldaps_etc_0_blh'] * feat_table['ldaps_wind_shear']

dt_index = feat_table.index
feat_table['hour_sin'] = np.sin(2*np.pi*dt_index.hour/24)
feat_table['hour_cos'] = np.cos(2*np.pi*dt_index.hour/24)
feat_table['month_sin'] = np.sin(2*np.pi*dt_index.month/12)
feat_table['month_cos'] = np.cos(2*np.pi*dt_index.month/12)
feat_table['doy_sin'] = np.sin(2*np.pi*dt_index.dayofyear/365)
feat_table['doy_cos'] = np.cos(2*np.pi*dt_index.dayofyear/365)

In [165]:
feat_table = feat_table.reset_index().rename(columns={'forecast_kst_dtm': 'kst_dtm'})

In [ ]:
feat_table.to_csv('data/derived_features_train.csv', index=False, encoding='utf-8-sig')

# 모델링까지 해보자